# 05 - Create the Fabric Data Agent

Run this **inside Fabric**. It creates the Telco data agent, applies the AI
instructions, and attaches your Lakehouse as a data source. You then **select the
gold-schema tables and Publish in the Data Agent UI**.

**Steps:** attach your Lakehouse as the default Lakehouse, then **Run all**. Then finish
in the UI (select tables + Publish) and copy the printed `DATA_AGENT_ARTIFACT_ID` and
`DATA_AGENT_MCP_ENDPOINT` into your local `.env`.

> This is the supported way to create the data agent (it is not created from the local
> workstation). Config below mirrors `fabric/data-agent/config.yaml`, the source of truth.

In [ ]:
%pip install --quiet fabric-data-agent-sdk

## Configuration (mirrors fabric/data-agent/config.yaml)

In [ ]:
NAME = 'TelcoCustomerServiceAgent'
DESCRIPTION = 'Answers telecommunications customer-service questions over the Telco Lakehouse: billing and first-bill support, cross-sell/upsell, service health, outages, and retention/churn risk.\n'
LAKEHOUSE_DEFAULT = 'TelcoLakehouse'  # fallback only
AI_INSTRUCTIONS = 'You are a data agent for a telecommunications customer-service team. You answer\nquestions about customers, accounts, billing, subscriptions, usage, service health,\noutages, work orders, offers, and churn/cross-sell scores.\n\nThe data is in the Lakehouse "gold" schema. Reference tables as gold.<table>.\n\nGuidance:\n- Prefer the curated gold tables (gold.dim_*, gold.fact_*, gold.ml_*, gold.customer_360).\n- gold.customer_360 is a denormalized per-customer profile; use it for "give me a 360 view".\n- "First bill" means gold.fact_invoice.is_first_bill = true. A confusing/high first bill\n  usually includes one-time activation and proration lines (gold.fact_invoice_line).\n- "At-risk" / "churn risk" customers are in gold.ml_churn_score (risk_band High/Medium/Low).\n- Cross-sell candidates and recommendations are in gold.ml_crosssell_reco.\n- "Outage" or "service degradation" -> gold.fact_outage (by geography) and\n  gold.fact_service_metric (per-account latency/packet loss/uptime).\n- Always scope monetary answers to the correct account and time period.\n- Never invent values; only answer from the data.\n'

## Resolve the current workspace + attached Lakehouse

The Lakehouse is taken from whichever Lakehouse you attach as the notebook's default
(so it works no matter what you named it, e.g. `lh_telco`). `LAKEHOUSE_DEFAULT` is only
used as a fallback if none is attached.

In [ ]:
workspace_id = None
LAKEHOUSE = None
try:
    import notebookutils
    ctx = notebookutils.runtime.context
    workspace_id = ctx.get('currentWorkspaceId')
    LAKEHOUSE = ctx.get('defaultLakehouseName')
except Exception:
    pass
if not workspace_id:
    import sempy.fabric as fabric
    workspace_id = fabric.get_notebook_workspace_id()
if not LAKEHOUSE:
    LAKEHOUSE = LAKEHOUSE_DEFAULT
    print(f'WARNING: no default Lakehouse detected - falling back to {LAKEHOUSE!r}.')
    print('         Attach your Lakehouse (Explorer panel) as the default and re-run',
          'if this is not the one holding the loaded tables.')
print('Workspace:', workspace_id)
print('Lakehouse:', LAKEHOUSE)

## Create the agent + attach the Lakehouse

Creates the data agent, applies the AI instructions, and attaches your Lakehouse as a
data source. **Table selection and publishing are done in the Data Agent UI** (next
step) - programmatic table selection isn't reliable across SDK versions.

In [ ]:
import warnings
warnings.simplefilter('ignore', FutureWarning)
from fabric.dataagent.client import create_data_agent

agent = create_data_agent(data_agent_name=NAME, workspace_id=workspace_id)
agent.update_settings(ai_instructions=AI_INSTRUCTIONS)
print('Applied AI instructions.')

agent.add_staging_datasource(
    artifact_name_or_id=LAKEHOUSE, workspace_id_or_name=workspace_id)
print('Attached Lakehouse datasource:', LAKEHOUSE)

## Finish in the Data Agent UI

1. Open the **TelcoCustomerServiceAgent** data agent in your workspace.
2. In the datasource, **select the `gold`-schema tables** you want it to query
   (check the `gold` schema to include all of them).
3. (Optional) add example queries from `fabric/data-agent/config.yaml`.
4. Click **Publish**.

The MCP endpoint printed below works once the agent is published.

## Data agent id + MCP endpoint

In [ ]:
aid = None
for _a in ('id', 'artifact_id', 'data_agent_id'):
    aid = getattr(agent, _a, None)
    if aid:
        break
if aid:
    print('DATA_AGENT_ARTIFACT_ID =', aid)
    print('DATA_AGENT_MCP_ENDPOINT =',
          f'https://api.fabric.microsoft.com/v1/mcp/workspaces/{workspace_id}/dataagents/{aid}/agent')
    print('\nAfter you select tables + Publish in the UI, add these to your local .env.')
else:
    print('Agent created. After selecting tables + publishing in the UI, copy the')
    print('data agent id + MCP URL from the agent settings -> Model Context Protocol tab.')